# 13. Profiling and Bottleneck Analysis | 性能分析与瓶颈定位

**难度：** Medium | **环境：** GPU optional | **标签：** `系统性能`, `Profiling`, `瓶颈分析` | **目标人群：** 需要定位训练或推理性能瓶颈的学习者

---

## 本节导读

性能记录中的计算量、访存字节、阶段耗时、数据搬运、同步等待和启动开销，需要先放回同一个固定工作负载，才能用于解释瓶颈。

本节建立“原始记录 → 派生指标 → 候选瓶颈 → 判别实验”的归因链。完成后，你应能区分计算、访存、搬运、同步和启动约束，并为候选解释设计单变量验证。



**关键词：** 性能剖析（profiling）、算术强度、机器平衡点、瓶颈归因

![本节概念关系](../docs/public/01_Hardware_Math_and_Systems/13_profiling_evidence_map.svg)

---


## 前置阅读

**导语：** 先建立 GPU 计算与访存的资源模型，并从 Part 00 获得固定工作负载下的阶段耗时与追踪记录。

- [Part 01 · 03 GPU 架构与显存](./03_GPU_Architecture_and_Memory.ipynb)
- [Part 00 · 17 PyTorch 性能分析基础](../00_Prerequisites/17_PyTorch_Profiling_Basics.ipynb)
- [Part 01 · 02 参数量与 FLOPs](./02_LLM_Params_and_FLOPs.ipynb)
---

## 证据从哪里来

不同工具提供不同粒度的原始证据。本节不重复采集过程，而是使用这些证据做资源归因。

| 证据层级 | 典型来源 | 本节使用的字段 |
|:---|:---|:---|
| 阶段与算子 | `torch.profiler` | 阶段耗时、算子耗时、调用次数 |
| 系统时间线 | Nsight Systems | 搬运、同步、启动与暴露等待 |
| Kernel 指标 | Nsight Compute | FLOPs、DRAM 字节、实际吞吐与停顿原因 |


## Q1：怎样把原始记录转换成可比较的资源指标？

原始记录中的 FLOPs、DRAM 字节和 Kernel 时间需要先换成统一单位。算术强度表示每搬运一个字节完成多少次浮点运算；实际计算吞吐和实际带宽表示当前 Kernel 在给定时间内交付了多少工作。

| 派生指标 | 计算关系 | 回答的问题 |
|:---|:---|:---|
| 算术强度 | FLOPs / DRAM bytes | 这段计算更依赖计算还是数据搬运？ |
| 实际计算吞吐 | FLOPs / kernel time | 当前交付了多少 TFLOP/s？ |
| 实际带宽 | DRAM bytes / kernel time | 当前消耗了多少 GB/s？ |
| 暴露时间占比 | 某类等待 / 总时间 | 搬运、同步或启动是否位于关键路径？ |


### Q1 小验证：从同一条记录派生资源指标

固定一条 Kernel 与系统时间线记录，计算算术强度、实际吞吐、实际带宽及各类暴露时间占比。

In [ ]:
def derive_resource_metrics(record):
    """把 Kernel 工作量与暴露时间转换成统一资源指标。"""
    required = {'flops', 'dram_bytes', 'kernel_ms', 'transfer_ms', 'sync_ms', 'launch_ms'}
    if set(record) != required:
        raise ValueError(f'record 必须且只能包含 {sorted(required)}')
    if record['flops'] <= 0 or record['dram_bytes'] <= 0 or record['kernel_ms'] <= 0:
        raise ValueError('flops、dram_bytes 和 kernel_ms 必须大于 0')
    if any(record[name] < 0 for name in ('transfer_ms', 'sync_ms', 'launch_ms')):
        raise ValueError('暴露时间不能为负数')
    total_ms = sum(record[name] for name in ('kernel_ms', 'transfer_ms', 'sync_ms', 'launch_ms'))
    return {
        'arithmetic_intensity_flop_per_byte': record['flops'] / record['dram_bytes'],
        'achieved_tflops': record['flops'] / (record['kernel_ms'] / 1000) / 1e12,
        'achieved_dram_gbps': record['dram_bytes'] / (record['kernel_ms'] / 1000) / 1e9,
        'total_ms': total_ms,
        'exposed_share': {
            name: record[f'{name}_ms'] / total_ms
            for name in ('transfer', 'sync', 'launch')
        },
    }

profile_record = {
    'flops': 4.0e10, 'dram_bytes': 8.0e9, 'kernel_ms': 20.0,
    'transfer_ms': 3.0, 'sync_ms': 2.0, 'launch_ms': 1.0,
}
resource_metrics = derive_resource_metrics(profile_record)
print(resource_metrics)
assert resource_metrics['arithmetic_intensity_flop_per_byte'] == 5.0
assert round(resource_metrics['achieved_tflops'], 1) == 2.0
assert round(resource_metrics['achieved_dram_gbps'], 1) == 400.0


## Q2：机器平衡点与暴露时间怎样形成瓶颈假设？

机器平衡点等于峰值计算吞吐除以峰值带宽，单位也是 FLOP/byte。Kernel 的算术强度低于该点时，Roofline 上限首先由带宽决定；高于该点时，首先由计算吞吐决定。系统时间线中的搬运、同步和启动时间还可能覆盖 Kernel 内部的计算或访存限制，因此要把两类证据放在一起。

| 证据 | 归因对象 | 仍需排除的情况 |
|:---|:---|:---|
| 算术强度 < 机器平衡点，Kernel 时间主导 | 访存路径 | 布局、缓存命中和实际带宽是否异常 |
| 算术强度 ≥ 机器平衡点，Kernel 时间主导 | 计算路径 | Tensor Core、占用率和指令效率是否达标 |
| H2D / D2H 暴露时间主导 | 数据搬运 | 输入准备、固定内存和异步拷贝是否生效 |
| 同步或启动时间主导 | 调度路径 | 依赖、碎片化 Kernel 和 CPU 发射开销 |


### Q2 小验证：比较机器平衡点与关键路径时间

用设备峰值计算机器平衡点，再将 Kernel 内部上限与系统时间线的最大暴露项合成第一轮归因。

In [ ]:
def attribute_bottleneck(record, peak_tflops, peak_dram_gbps):
    """结合 Roofline 上限和暴露时间，返回第一轮瓶颈假设。"""
    if peak_tflops <= 0 or peak_dram_gbps <= 0:
        raise ValueError('设备峰值必须大于 0')
    metrics = derive_resource_metrics(record)
    machine_balance = peak_tflops * 1000 / peak_dram_gbps
    kernel_bound = (
        'memory'
        if metrics['arithmetic_intensity_flop_per_byte'] < machine_balance
        else 'compute'
    )
    exposed_ms = {
        f'kernel_{kernel_bound}': record['kernel_ms'],
        'transfer': record['transfer_ms'],
        'synchronization': record['sync_ms'],
        'launch': record['launch_ms'],
    }
    primary = max(exposed_ms, key=exposed_ms.get)
    return {
        'machine_balance_flop_per_byte': machine_balance,
        'kernel_bound': kernel_bound,
        'exposed_ms': exposed_ms,
        'primary_hypothesis': primary,
    }

memory_case = attribute_bottleneck(profile_record, peak_tflops=120, peak_dram_gbps=3000)
transfer_record = {**profile_record, 'kernel_ms': 6.0, 'transfer_ms': 14.0}
transfer_case = attribute_bottleneck(transfer_record, peak_tflops=120, peak_dram_gbps=3000)
print('memory_case =', memory_case)
print('transfer_case =', transfer_case)
assert memory_case['kernel_bound'] == 'memory'
assert memory_case['primary_hypothesis'] == 'kernel_memory'
assert transfer_case['primary_hypothesis'] == 'transfer'


## Q3：如何用判别实验验证或推翻瓶颈假设？

归因结果决定下一项实验，而不是直接决定最终优化方案。判别实验需要保持工作负载不变，只改变一个能够影响当前假设的条件，并同时记录目标指标和代价指标。如果目标指标没有按预期变化，应回到时间线修改假设。

| 假设 | 单一变化 | 主要观察 | 证伪条件 |
|:---|:---|:---|:---|
| Kernel 访存受限 | 改变 tile、融合或布局之一 | DRAM bytes、Kernel 时间 | 字节数下降但时间无显著变化 |
| Kernel 计算受限 | 改变 dtype 或计算 Kernel 之一 | TFLOP/s、质量误差 | 计算吞吐提高但端到端时间不变 |
| 数据搬运受限 | 只改变 pinned memory 或异步搬运 | 暴露搬运时间 | 搬运重叠后关键路径没有缩短 |
| 同步或启动受限 | 只改变融合、Graph 或批量组织之一 | launch 数、同步等待 | 事件减少但阶段时间不变 |


### Q3 小验证：为归因结果生成单变量判别实验

将 Q2 的主假设转换成变化变量、观察指标和证伪条件，避免一次修改多个策略。

In [ ]:
def build_discrimination_experiment(attribution):
    """把单个主假设转换成可证伪的单变量实验。"""
    plans = {
        'kernel_memory': {
            'single_change': 'tile_or_layout',
            'target_metrics': ['dram_bytes', 'kernel_ms'],
            'falsified_when': 'dram_bytes_down_but_kernel_ms_unchanged',
        },
        'kernel_compute': {
            'single_change': 'dtype_or_compute_kernel',
            'target_metrics': ['achieved_tflops', 'kernel_ms', 'quality'],
            'falsified_when': 'tflops_up_but_end_to_end_unchanged',
        },
        'transfer': {
            'single_change': 'pinned_or_async_transfer',
            'target_metrics': ['exposed_transfer_ms', 'stage_ms'],
            'falsified_when': 'transfer_overlap_up_but_stage_ms_unchanged',
        },
        'synchronization': {
            'single_change': 'dependency_or_sync_placement',
            'target_metrics': ['sync_ms', 'stage_ms'],
            'falsified_when': 'sync_ms_down_but_stage_ms_unchanged',
        },
        'launch': {
            'single_change': 'fusion_or_graph_capture',
            'target_metrics': ['launch_count', 'launch_ms', 'stage_ms'],
            'falsified_when': 'launches_down_but_stage_ms_unchanged',
        },
    }
    hypothesis = attribution['primary_hypothesis']
    if hypothesis not in plans:
        raise ValueError(f'不支持的瓶颈假设: {hypothesis}')
    return {'hypothesis': hypothesis, **plans[hypothesis]}

memory_plan = build_discrimination_experiment(memory_case)
transfer_plan = build_discrimination_experiment(transfer_case)
print('memory_plan =', memory_plan)
print('transfer_plan =', transfer_plan)
assert memory_plan['single_change'] == 'tile_or_layout'
assert transfer_plan['target_metrics'] == ['exposed_transfer_ms', 'stage_ms']


---
## 相关阅读
从资源归因继续进入训练、推理或端到端项目，用固定工作负载验证判别实验。
- [PyTorch Profiler 官方文档](https://pytorch.org/docs/stable/profiler.html)
- [Part 02 · 73 训练性能分析](../02_PyTorch_Algorithms/73_Training_Performance_Analysis.ipynb)
- [Part 02 · 74 Profiling 驱动的端到端优化](../02_PyTorch_Algorithms/74_Profiling_Driven_End_to_End_Optimization.ipynb)
---